<a href="https://colab.research.google.com/github/kanakamvasundhara/research-paper-rag/blob/main/research_paper_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install required packages
!pip install pypdf faiss-cpu sentence-transformers transformers

# 2. Import libraries
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

# 3. Load and Extract PDF text
pdf_file = "/content/nlp_application.pdf"
reader = PdfReader(pdf_file)
pages = []

for page in reader.pages:
    text = page.extract_text()
    if text:
        pages.append(text)

print("Number of pages extracted:", len(pages))

# 4. Chunk the extracted text
chunks = []
chunk_size = 500
overlap = 100

for page_no, text in enumerate(pages, start=1):
    words = text.split()
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.strip()) > 50:
            chunks.append({
                "text": chunk,
                "page": page_no
            })

print("Total chunks created:", len(chunks))

# 5. Generate text embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

# 6. Store embeddings in FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))
print("Vectors stored in FAISS:", index.ntotal)

# 7. Initialize the text2text generator (Fixed pipeline for FLAN-T5)
generator = pipeline(
    "text-generation", # Changed from "text2text-generation" to "text-generation"
    model="google/flan-t5-base",
    max_new_tokens=250
)

# 8. Define the QA Function (Fixed syntax loop and optimized context size)
def ask_question(question, k=2):
    # Encode user question
    query_embedding = model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS index for closest matching chunks
    distances, indices = index.search(query_embedding, k)
    retrieved = []
    for idx in indices[0]:
        if idx < len(chunks):
            retrieved.append(chunks[idx])

    # Construct the RAG prompt context
    context = "\n\n".join(
        f"[Page {item['page']}]\n{item['text']}"
        for item in retrieved
    )

    prompt = f"""
Answer the question using ONLY the information given in the context.
If the answer is not available in the context, say: "Information not found in the research paper."

Give a short and clear answer.

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate response from FLAN-T5
    response = generator(prompt)[0]["generated_text"]

    print("\nANSWER:")
    print(response)

    print("\nSOURCES:")
    for item in retrieved:
        print(f"- Page {item['page']}")

    return response


In [ ]:
ask_question("What is the main finding of the paper?")
